# Zero shot batch integration tutorial

# 🧬 Overview: Aims & Key Findings

## 🎯 Aims

This study is designed to demonstrate:

1. **Generalizability & Transferability** – The ability of pre‑trained representations to generalize across spatial and single‑cell contexts.
2. **Feature Robustness** – The reliability of features learned specifically from the **Xenium** spatial transcriptomics dataset.
3. **Biological Signal Conservation** – The model’s capacity to preserve and leverage biologically meaningful signals.

---

## 🔬 Key Results

Pre‑training on a **spatially‑aware** model yields performance that is:

- **Comparable to**, and in some tasks **slightly superior to**, leading single‑cell foundation models (e.g., *scGPT*, *scFoundation*), despite being trained on a distinct data modality.
- Achieved with a substantially smaller gene set, underscoring the efficiency and robustness of spatial pre‑training.

These findings highlight the **enhanced capability** of our spatially‑aware pre‑training scheme and its strong potential for transfer to diverse transcriptomic applications.

# Dataset Overview

This notebook uses the dataset associated with the publication [Lotfollahi, M. et al.,](https://doi.org/10.1038/s41587-021-01001-7):

> **DOI: [10.6084/m9.figshare.31884799](https://doi.org/10.6084/m9.figshare.31884799)**

The dataset is permanently archived on **Figshare** under the above DOI. For convenience during this Colab session, the data is also made available via a public Google Drive folder.

## Accessing the Data

### Option 1: Direct Download from Google Drive (Recommended for Colab)

You can download the dataset directly using the following public Google Drive link:

[https://drive.google.com/drive/folders/12-vuL-gx_wKiLcZlT_ie_mfZEwiNzeND](https://drive.google.com/drive/folders/12-vuL-gx_wKiLcZlT_ie_mfZEwiNzeND)

If you prefer to work with the files within Colab, mount your Google Drive and copy the folder:

```python
from google.colab import drive
drive.mount('/content/drive')

# Copy the folder to your Colab environment (adjust destination as needed)
!cp -r "/content/drive/MyDrive/path-to-dataset-folder" "/content/dataset"
```
### Option 2: Direct Download via gdown package
```shell
! gdown 1hb0TEQBIOWtGpEJXyBS6Ah6EEHEDR6Bm
```

In [13]:
! gdown 1hb0TEQBIOWtGpEJXyBS6Ah6EEHEDR6Bm

Downloading...
From: https://drive.google.com/uc?id=1hb0TEQBIOWtGpEJXyBS6Ah6EEHEDR6Bm
To: /content/covid_subsampled.h5ad
100% 25.0M/25.0M [00:00<00:00, 28.9MB/s]


## 🧠 Model Checkpoint Loading

The pre‑trained model checkpoint is available for download from the following link:

🔗 **[Download Checkpoint](https://doi.org/10.6084/m9.figshare.31146238)**

For seamless integration within Colab, you can download it directly using `gdown`:
```python
!gdown 16j_GYeqhWYMM1kpagw4HQlap_4z6MEu6
```

In [14]:
!gdown 16j_GYeqhWYMM1kpagw4HQlap_4z6MEu6

Downloading...
From (original): https://drive.google.com/uc?id=16j_GYeqhWYMM1kpagw4HQlap_4z6MEu6
From (redirected): https://drive.google.com/uc?id=16j_GYeqhWYMM1kpagw4HQlap_4z6MEu6&confirm=t&uuid=03d720f1-c730-41a3-9911-460e9a1a1695
To: /content/SpatialFormer_single_input_5k.ckpt
100% 721M/721M [00:06<00:00, 108MB/s] 


In [31]:
# @title Install Dependencies
# @markdown This cell installs PyTorch (CUDA 12.1), CMake, and spatialformer.
# @markdown It may take 2-5 minutes.
import subprocess
import sys

# 1. Install CMake (Required for build)
!apt-get install -y cmake

# 2. Install PyTorch (Ensure CUDA 12.1 compatibility)
# Colab usually has torch pre-installed, but we force the correct version to match spatialformer requirements
!pip install torch==2.3.1 torchvision==0.18.1 torchaudio==2.3.1 --index-url https://download.pytorch.org/whl/cu121
# 3. Install spatialformer
!pip install spatialformer==0.0.20
!pip install gdown
!pip install scib
print("✅ Installation Complete!")


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
cmake is already the newest version (3.22.1-1ubuntu1.22.04.2).
0 upgraded, 0 newly installed, 0 to remove and 45 not upgraded.
Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 104.5 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.
✅ Installation Complete!


In [16]:
# @title (Optional) Install FlashAttention for Speed if you have A100/H100 available
! pip install https://github.com/Dao-AILab/flash-attention/releases/download/v2.6.3/flash_attn-2.6.3+cu123torch2.3cxx11abiFALSE-cp312-cp312-linux_x86_64.whl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.3/187.3 MB 12.0 MB/s eta 0:00:00


## COVID-19 dataset Integration

In [27]:
# @title Import any dependencies
import scanpy as sc
import matplotlib.pyplot as plt
import sys
import spatialformer as sp
import torch
import scib
import numpy as np
import os
import sys
from pathlib import Path
sys.path = [str(p) if isinstance(p, Path) else p for p in sys.path]


In [18]:
# @title Loading the datasets
covid_data = sc.read_h5ad("/content/covid_subsampled.h5ad")

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


### Getting the embeddings

In [19]:
! gdown 1xEhe1l0Nhmi1jLvvjPqWDIL3t4iYn5jw

Downloading...
From (original): https://drive.google.com/uc?id=1xEhe1l0Nhmi1jLvvjPqWDIL3t4iYn5jw
From (redirected): https://drive.google.com/uc?id=1xEhe1l0Nhmi1jLvvjPqWDIL3t4iYn5jw&confirm=t&uuid=345e55d7-ca41-4976-8e6b-c839a31921ea
To: /content/SpatialFormer_single_input.ckp
100% 643M/643M [00:14<00:00, 44.9MB/s]


In [30]:
%%time
method = "cls"
tissue = "Lung"
condition = "Disease"
model_ckp_path = "./SpatialFormer_single_input_5k.ckpt"
use_flash_attn = True # Depends on whether you install the FlashAttention, if installed -> "True", "False" instead.
batch_size = 16
os.environ['TORCH_USE_CUDA_DSA'] = "1"
embed_adata = sp.tl.embed_data(
                            adata = covid_data,
                            tissue = tissue,
                            condition = condition,
                            method = method,
                            model_ckp_path = model_ckp_path,
                            batch_size = batch_size,
                            mode = "single",
                            use_flash_attn = use_flash_attn,
                            num_workers = 32,
                            version = "v1"
                            )

Spatialformer - INFO - Redirected the GraphSAGE embedding_path...
Spatialformer - INFO - Loading the SpatialFormer model...


INFO:lightning_fabric.utilities.seed:Seed set to 42


RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1.
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


### Visualization of the Spatialformer embeddings

In [21]:
cell_type_key = "celltype"
sc.pp.neighbors(embed_adata, use_rep="X_SpaF")
sc.tl.umap(embed_adata)
with plt.rc_context({"figure.figsize": (7, 5), "figure.dpi": (300)}):
    sc.pl.umap(embed_adata,
               color=[cell_type_key],
               frameon=False,
               wspace=0.4,
               title=["COVID19 - SpatialFormer zero-shot: cell type"],
               save = "Spatialformer_COVID19_zero-shot_celltype.png"
              )

NameError: name 'embed_adata' is not defined

### Visualizing the batch

In [ ]:
batch_key = "str_batch"
with plt.rc_context({"figure.figsize": (7, 5), "figure.dpi": (300)}):
    sc.pl.umap(embed_adata,
               color=[batch_key],
               frameon=False,
               title=["COVID19 - SpatialFormer zero-shot: batch label"],
               save = "Spatialformer_COVID19_zero-shot_batch.png"
               )

### Computing the score of the metrics - batch correction effect

In [ ]:
def scib_eval(adata, batch_key, cell_type_key, embed_key):
    results = scib.metrics.metrics(
        adata,
        adata_int=adata,
        batch_key=batch_key,
        label_key=cell_type_key,
        embed=embed_key,
        isolated_labels_asw_=False,
        silhouette_=True,
        hvg_score_=False,
        graph_conn_=True,
        pcr_=True,
        isolated_labels_f1_=False,
        trajectory_=False,
        nmi_=True,  # use the clustering, bias to the best matching
        ari_=True,  # use the clustering, bias to the best matching
        cell_cycle_=False,
        kBET_=False,
        ilisi_=False,
        clisi_=False,
    )
    result_dict = results[0].to_dict()

    # compute avgBIO metrics
    result_dict["avg_bio"] = np.mean(
        [
            result_dict["NMI_cluster/label"],
            result_dict["ARI_cluster/label"],
            result_dict["ASW_label"],
        ]
    )

    # compute avgBATCH metrics
    result_dict["avg_batch"] = np.mean(
        [
            result_dict["graph_conn"],
            result_dict["ASW_label/batch"],
        ]
    )

    result_dict = {k: v for k, v in result_dict.items() if not np.isnan(v)}

    return result_dict

In [ ]:

scib_result_dict = scib_eval(
    embed_adata,
    batch_key=batch_key,
    cell_type_key=cell_type_key,
    embed_key="X_SpaF",
)
print("AvgBIO: {:.4f}".format(scib_result_dict["avg_bio"]))
print("AvgBATCH: {:.4f}".format(scib_result_dict["avg_batch"]))

#For 28000 ckp:  
AvgBIO: 0.5012  
AvgBATCH: 0.8716

### Comparison with HGV+PCs

In [ ]:
sc.pp.pca(covid_data, n_comps=40)
sc.pp.neighbors(covid_data, use_rep="X_pca")
sc.pp.neighbors(covid_data, use_rep="X_pca")
sc.tl.umap(covid_data)
sc.pl.umap(
    covid_data,
    color=[cell_type_key, batch_key],
    frameon=False,
    wspace=0.4,
    title=["HVG+PCs: cell type", "HVG+PCs: batch label"],
    ncols=1,
    save = "COVID19_PCA_celltype.png"
)

In [ ]:
scib_result_dict = scib_eval(
    covid_data,
    batch_key=batch_key,
    cell_type_key=cell_type_key,
    embed_key="X_pca",
)
print("AvgBIO: {:.4f}".format(scib_result_dict["avg_bio"]))
print("AvgBATCH: {:.4f}".format(scib_result_dict["avg_batch"]))

# Lung-kim dataset

In [ ]:
import anndata

smaple_data_path = '/scratch/project_465001027/Spatialformer/downstream/zero-shot_batch_correction/data/Kim2020_Lung.h5ad'
adata = sc.read_h5ad(smaple_data_path)

gene_col = "gene_name"
cell_type_key = "cell_type"
batch_key = "sample"
# Remove unannotated cells:
celltype_id_labels = adata.obs[cell_type_key].astype("category").cat.codes.values
adata = adata[celltype_id_labels >= 0]
org_adata = adata.copy()


tissue = "Lung"
condition = "Disease"
method = "cls"
model_ckp_path = "/scratch/project_465001027/Spatialformer/output/checkpoints/step=0028000-train_total_loss=-1.7847-val_total_loss=0.0000.ckpt"

# highly variable genes
# sc.pp.highly_variable_genes(adata, n_top_genes=N_HVG, flavor='seurat_v3')
# adata = adata[:, adata.var['highly_variable']]
embed_adata = sp.tl.embed_data(org_adata,
                              tissue,
                              condition,
                               method,
                            model_ckp_path,
                            batch_size,
                            mode = "single",
                            threshold = 0.7,
                            )

In [ ]:
embed_adata

In [ ]:
sc.pp.neighbors(embed_adata, use_rep="X_SpaF")
sc.tl.umap(embed_adata)
with plt.rc_context({"figure.figsize": (7, 5), "figure.dpi": (300)}):
    sc.pl.umap(embed_adata,
               color=[cell_type_key, batch_key],
               frameon=False,
               wspace=0.4,
               save = "kim_lung_Spatialformer.png",
               title=["SpaF zero-shot: cell type", "spaF zero-shot: batch label"])


scib_result_dict = scib_eval(
    embed_adata,
    batch_key=batch_key,
    cell_type_key=cell_type_key,
    embed_key="X_SpaF"

)

In [ ]:
print("AvgBIO: {:.4f}".format(scib_result_dict["avg_bio"]))
print("AvgBATCH: {:.4f}".format(scib_result_dict["avg_batch"]))

# PCA results

In [ ]:
sc.pp.pca(embed_adata, n_comps=40)
sc.pp.neighbors(embed_adata, use_rep="X_pca")
sc.tl.umap(embed_adata)
with plt.rc_context({"figure.figsize": (7, 5), "figure.dpi": (300)}):
    sc.pl.umap(embed_adata,
               color=[cell_type_key, batch_key],
               frameon=False,
               wspace=0.4,
               title=["HVG+PCs: cell type", "HVG+PCs: batch label"],
               save = "PCA_kim_lung.png")

In [ ]:
scib_result_dict = scib_eval(
    embed_adata,
    batch_key=batch_key,
    cell_type_key=cell_type_key,
    embed_key="X_pca",
)

print("AvgBIO: {:.4f}".format(scib_result_dict["avg_bio"]))
print("AvgBATCH: {:.4f}".format(scib_result_dict["avg_batch"]))

In [ ]:
import gseapy as gp
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

filtered_degs = all_filtered_degs[all_filtered_degs["cluster"] == str(cluster)]
degs = list(filtered_degs.set_index(["gene"]).index)
deg_df = filtered_degs.set_index(["gene"])
results = gp.enrichr(gene_list=degs,
 gene_sets=['GO_Cellular_Component_2021'],
 organism='Human',
 outdir=None,
 cutoff=0.5
 )
from gseapy.plot import barplot, dotplot
barplot(results.res2d,title=f'GO Cellular_Component cluster {cluster}',color = 'r')

In [ ]:
pair_gene_list = list(set([gene for pair in embed_adata_cd4_flt.obs["Gene_Pairs"][0] for gene in pair]))

results = gp.enrichr(gene_list=pair_gene_list,
 gene_sets=['GO_Cellular_Component_2021'],
 organism='Human',
 outdir=None,
 cutoff=0.5
 )
from gseapy.plot import barplot, dotplot
barplot(results.res2d,title=f'GO Cellular_Component cluster paired genes',color = 'r')